#### ─── Notebook 05: Analytics & KPIs ──────────────────────────────
#### Reads from Silver + Gold tables — no file paths needed

In [0]:
SILVER_TABLE  = "workspace.new_project.silver_online_retail"
GOLD_COUNTRY  = "workspace.new_project.gold_revenue_by_country"
GOLD_PRODUCTS = "workspace.new_project.gold_top_products"
GOLD_MONTHLY  = "workspace.new_project.gold_monthly_trend"
GOLD_RFM      = "workspace.new_project.gold_customer_rfm"

from pyspark.sql.functions import (
    col, sum, count, avg, max, min,
    round, countDistinct, month, year,
    hour, desc, asc, when, lit,
    dense_rank, lag, percent_rank
)
from pyspark.sql.window import Window

silver = spark.read.table(SILVER_TABLE)

print("Tables loaded")

Tables loaded


#### ── KPI 1: Executive Dashboard — Top-Line Numbers ────────────────

In [0]:
print("\n" + "="*55)
print("  EXECUTIVE DASHBOARD — TOP LINE KPIs")
print("="*55)

executive_kpis = silver.agg(
    round(sum("total_amount"), 2).alias("total_revenue_gbp"),
    countDistinct("invoice_id").alias("total_orders"),
    countDistinct("customer_id").alias("unique_customers"),
    countDistinct("stock_code").alias("unique_products"),
    round(avg("total_amount"), 2).alias("avg_order_value"),
    round(avg("quantity"), 2).alias("avg_items_per_order"),
    round(sum("total_amount") / countDistinct("customer_id"), 2).alias("revenue_per_customer")
)

display(executive_kpis)

# Save as Gold table for dashboards
executive_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.new_project.gold_executive_kpis")

print("✅ gold_executive_kpis saved")


  EXECUTIVE DASHBOARD — TOP LINE KPIs


total_revenue_gbp,total_orders,unique_customers,unique_products,avg_order_value,avg_items_per_order,revenue_per_customer
1.728736419E7,36967,5878,4631,22.48,13.61,2941.03


✅ gold_executive_kpis saved


#### # ── KPI 2: Revenue by Country — Top 10

In [0]:
print("\n" + "="*55)
print("  REVENUE BY COUNTRY — TOP 10")
print("="*55)

country_df = spark.read.table(GOLD_COUNTRY)

# Add revenue share % column
total_rev = country_df.agg(sum("total_revenue")).collect()[0][0]

country_with_share = country_df \
    .withColumn("revenue_share_pct",
                round((col("total_revenue") / total_rev) * 100, 2)) \
    .orderBy(desc("total_revenue"))

display(country_with_share.limit(10))


  REVENUE BY COUNTRY — TOP 10


country,total_revenue,total_orders,unique_customers,avg_order_value,revenue_share_pct
UNITED KINGDOM,1.431078924E7,689970,5350,20.74,82.78
EIRE,616544.54,15561,5,39.62,3.57
NETHERLANDS,553805.51,5079,22,109.04,3.2
GERMANY,423823.42,16417,107,25.82,2.45
FRANCE,348570.34,13472,95,25.87,2.02
AUSTRALIA,169250.26,1788,15,94.66,0.98
SPAIN,108122.23,3636,41,29.74,0.63
SWITZERLAND,100061.94,3005,22,33.3,0.58
SWEDEN,91515.82,1317,19,69.49,0.53
DENMARK,68539.44,777,12,88.21,0.4


# ── KPI 3: Monthly Revenue Trend + Month-on-Month Growth 

In [0]:
print("\n" + "="*55)
print("  MONTHLY REVENUE TREND + MoM GROWTH")
print("="*55)

monthly_df = spark.read.table(GOLD_MONTHLY)

# Window: lag to get previous month revenue
window_prev = Window.partitionBy("year").orderBy("month")

monthly_with_growth = monthly_df \
    .withColumn("prev_month_revenue",
                lag("revenue", 1).over(window_prev)) \
    .withColumn("mom_growth_pct",
                round(
                    ((col("revenue") - col("prev_month_revenue"))
                     / col("prev_month_revenue")) * 100,
                2))

display(monthly_with_growth.orderBy("year", "month"))

monthly_with_growth.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.new_project.gold_monthly_growth")

print("✅ gold_monthly_growth saved")


  MONTHLY REVENUE TREND + MoM GROWTH


year,month,revenue,orders,customers,prev_month_revenue,mom_growth_pct
2009,12,679214.52,1512,955,null,null
2010,1,553701.62,1011,720,null,null
2010,2,502429.4,1104,772,553701.62,-9.26
2010,3,694418.17,1524,1057,502429.4,38.21
2010,4,589631.03,1329,942,694418.17,-15.09
2010,5,594628.38,1377,966,589631.03,0.85
2010,6,632224.16,1496,1041,594628.38,6.32
2010,7,587853.18,1381,928,632224.16,-7.02
2010,8,599770.88,1293,911,587853.18,2.03
2010,9,824950.1,1689,1145,599770.88,37.54


✅ gold_monthly_growth saved


####── KPI 4: Top 10 Products by Revenue

In [0]:
print("\n" + "="*55)
print("  TOP 10 PRODUCTS BY REVENUE")
print("="*55)

products_df = spark.read.table(GOLD_PRODUCTS)

display(
    products_df
    .orderBy("revenue_rank")
    .select("revenue_rank", "description", "stock_code",
            "total_revenue", "total_quantity")
    .limit(10)
)


  TOP 10 PRODUCTS BY REVENUE


revenue_rank,description,stock_code,total_revenue,total_quantity
1,REGENCY CAKESTAND 3 TIER,22423,276974.8,24069
2,WHITE HANGING HEART T-LIGHT HOLDER,85123A,245437.41,91185
3,"PAPER CRAFT , LITTLE BIRDIE",23843,168469.6,80995
4,Manual,M,143586.94,8949
5,JUMBO BAG RED RETROSPOT,85099B,134071.23,74105
6,POSTAGE,POST,124648.04,5235
7,ASSORTED COLOUR BIRD ORNAMENT,84879,123827.96,77924
8,PARTY BUNTING,47566,102814.03,23351
9,MEDIUM CERAMIC TOP STORAGE JAR,23166,81405.48,77907
10,PAPER CHAIN KIT 50'S CHRISTMAS,22086,75985.13,28165


#### ── KPI 5: Hourly Order Pattern

In [0]:
print("\n" + "="*55)
print("  HOURLY ORDER PATTERN (peak hours)")
print("="*55)

hourly = silver \
    .withColumn("hour_of_day", hour("invoice_date")) \
    .groupBy("hour_of_day") \
    .agg(
        countDistinct("invoice_id").alias("order_count"),
        round(sum("total_amount"), 2).alias("revenue"),
        round(avg("total_amount"), 2).alias("avg_order_value")
    ) \
    .orderBy("hour_of_day")

display(hourly)

hourly.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.new_project.gold_hourly_pattern")

print("✅ gold_hourly_pattern saved")


  HOURLY ORDER PATTERN (peak hours)


hour_of_day,order_count,revenue,avg_order_value
6,1,4.25,4.25
7,76,75745.57,71.93
8,979,524521.53,34.56
9,2548,1483845.33,36.24
10,4364,2314692.99,32.21
11,4608,2222482.78,23.28
12,6169,2674452.07,19.57
13,5418,2327572.06,18.64
14,4559,1951684.61,18.55
15,3992,1830902.56,21.55


✅ gold_hourly_pattern saved


#### ── KPI 6: Customer Segmentation — RFM Tiers

In [0]:
print("\n" + "="*55)
print("  CUSTOMER SEGMENTATION — RFM TIERS")
print("="*55)

rfm_df = spark.read.table(GOLD_RFM)

# Segment customers into tiers based on RFM values
rfm_segmented = rfm_df.withColumn(
    "customer_segment",
    when(
        (col("recency_days") <= 30) &
        (col("frequency") >= 5) &
        (col("monetary") >= 1000), "Champions"
    ).when(
        (col("recency_days") <= 60) &
        (col("frequency") >= 3), "Loyal Customers"
    ).when(
        (col("recency_days") <= 90) &
        (col("frequency") >= 2), "Potential Loyalists"
    ).when(
        (col("recency_days") > 180) &
        (col("frequency") == 1), "At Risk"
    ).when(
        col("recency_days") > 270, "Lost Customers"
    ).otherwise("Regular Customers")
)
# Summary per segment
segment_summary = rfm_segmented \
    .groupBy("customer_segment") \
    .agg(
        count("customer_id").alias("customer_count"),
        round(avg("monetary"), 2).alias("avg_spend"),
        round(avg("frequency"), 1).alias("avg_orders"),
        round(avg("recency_days"), 0).alias("avg_recency_days")
    ) \
    .orderBy(desc("customer_count"))

display(segment_summary)

rfm_segmented.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.new_project.gold_customer_segments")

print("✅ gold_customer_segments saved")


  CUSTOMER SEGMENTATION — RFM TIERS


customer_segment,customer_count,avg_spend,avg_orders,avg_recency_days
Regular Customers,1288,1353.53,3.7,131.0
At Risk,1152,329.91,1.0,473.0
Champions,1014,10179.74,18.8,12.0
Lost Customers,914,1519.01,3.9,430.0
Loyal Customers,874,2499.91,6.5,32.0
Potential Loyalists,636,1994.36,4.4,53.0


✅ gold_customer_segments saved


#### ── KPI 7: Repeat Purchase Rate

In [0]:
print("\n" + "="*55)
print("  REPEAT PURCHASE RATE")
print("="*55)

orders_per_customer = silver \
    .groupBy("customer_id") \
    .agg(countDistinct("invoice_id").alias("order_count"))

total_customers  = orders_per_customer.count()
repeat_customers = orders_per_customer.filter(col("order_count") >= 2).count()
one_time         = total_customers - repeat_customers
repeat_rate      = float(f"{(repeat_customers / total_customers) * 100:.1f}")

print(f"Total customers   : {total_customers:,}")
print(f"Repeat customers  : {repeat_customers:,}")
print(f"One-time buyers   : {one_time:,}")
print(f"Repeat rate       : {repeat_rate}%")

repeat_summary = spark.createDataFrame([
    ("Total customers",   total_customers),
    ("Repeat customers",  repeat_customers),
    ("One-time buyers",   one_time),
], ["metric", "value"]) \
    .withColumn("percentage",
                round(col("value") / total_customers * 100, 1))

display(repeat_summary)


  REPEAT PURCHASE RATE
Total customers   : 5,878
Repeat customers  : 4,255
One-time buyers   : 1,623
Repeat rate       : 72.4%


metric,value,percentage
Total customers,5878,100.0
Repeat customers,4255,72.4
One-time buyers,1623,27.6


#### ── KPI 8: Revenue Cumulative Running Total (YTD)

In [0]:
print("\n" + "="*55)
print("  CUMULATIVE REVENUE — YEAR TO DATE")
print("="*55)

monthly_df = spark.read.table(GOLD_MONTHLY)

window_ytd = Window \
    .partitionBy("year") \
    .orderBy("month") \
    .rowsBetween(Window.unboundedPreceding, 0)

ytd = monthly_df \
    .withColumn("cumulative_revenue",
                round(sum("revenue").over(window_ytd), 2)) \
    .withColumn("cumulative_orders",
                sum("orders").over(window_ytd)) \
    .orderBy("year", "month")

display(ytd)

ytd.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.new_project.gold_ytd_revenue")



  CUMULATIVE REVENUE — YEAR TO DATE


year,month,revenue,orders,customers,cumulative_revenue,cumulative_orders
2009,12,679214.52,1512,955,679214.52,1512
2010,1,553701.62,1011,720,553701.62,1011
2010,2,502429.4,1104,772,1056131.02,2115
2010,3,694418.17,1524,1057,1750549.19,3639
2010,4,589631.03,1329,942,2340180.22,4968
2010,5,594628.38,1377,966,2934808.6,6345
2010,6,632224.16,1496,1041,3567032.76,7841
2010,7,587853.18,1381,928,4154885.94,9222
2010,8,599770.88,1293,911,4754656.82,10515
2010,9,824950.1,1689,1145,5579606.92,12204


#### ── KPI 9: Best Day of Week for Orders

In [0]:
print("\n" + "="*55)
print("  ORDERS BY DAY OF WEEK")
print("="*55)

from pyspark.sql.functions import dayofweek, date_format

day_pattern = silver \
    .withColumn("day_of_week",
                date_format("invoice_date", "EEEE")) \
    .withColumn("day_num",
                dayofweek("invoice_date")) \
    .groupBy("day_num", "day_of_week") \
    .agg(
        countDistinct("invoice_id").alias("order_count"),
        round(sum("total_amount"), 2).alias("revenue")
    ) \
    .orderBy("day_num")

display(day_pattern)

day_pattern.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.new_project.gold_day_of_week_pattern")

print("✅ gold_day_of_week_pattern saved")


  ORDERS BY DAY OF WEEK


day_num,day_of_week,order_count,revenue
1,Sunday,4748,1747846.43
2,Monday,5754,2764062.25
3,Tuesday,6627,3310525.23
4,Wednesday,6649,3006172.05
5,Thursday,7772,3730519.93
6,Friday,5387,2718435.23
7,Saturday,30,9803.05


✅ gold_day_of_week_pattern saved


#### ── Final: List All Gold Tables Created

In [0]:
print("\n" + "="*55)
print("  ALL GOLD TABLES IN workspace.new_project")
print("="*55)

display(
    spark.sql("SHOW TABLES IN workspace.new_project")
    .filter(col("tableName").startswith("gold_"))
    .orderBy("tableName")
)

print("\n🎯 Analytics notebook complete!")


  ALL GOLD TABLES IN workspace.new_project


database,tableName,isTemporary
new_project,gold_customer_rfm,false
new_project,gold_customer_segments,false
new_project,gold_day_of_week_pattern,false
new_project,gold_executive_kpis,false
new_project,gold_hourly_pattern,false
new_project,gold_monthly_growth,false
new_project,gold_monthly_trend,false
new_project,gold_revenue_by_country,false
new_project,gold_top_products,false
new_project,gold_ytd_revenue,false



🎯 Analytics notebook complete!
